# Fine-Tuning Google Gemma 2B with QLoRA on Colab

**Objective:** Fine-tune the `google/gemma-2b` base model on the `tatsu-lab/alpaca` dataset using Parameter-Efficient Fine-Tuning (PEFT) with **QLoRA**. This reduces the memory footprint significantly, allowing the 2.5 billion parameter model to easily fit inside Google Colab's free 15GB T4 GPU.

*(Note: The original draft of this notebook referenced TinyLlama. This has been corrected to genuinely use Google's Gemma model while adopting best practices for security and efficiency.)*

### Training Pipeline (Conceptual)
Dataset → Prompt Formatting → Tokenizer → Gemma 2B (4-bit quantized) → Inject LoRA adapters → `SFTTrainer` → Fine-tuned model

### Notebook Phases
| Phase | Description |
|---|---|
| **Phase 1** | Environment Setup (Google Drive, Secrets, & Required Libraries) |
| **Phase 2** | Model & Tokenizer Initialization (4-bit Quantization, LoRA Setup) |
| **Phase 3** | Dataset Preparation (Loading alpaca data & Prompt Formatting) |
| **Phase 4** | Training Configuration (`TrainingArguments` & `SFTTrainer`) |
| **Phase 5** | Inference & Testing (Evaluating the Fine-Tuned Model) |

---
## Phase 1: Environment Setup
Initialize the Colab environment by mounting storage, handling secure API tokens, and installing the necessary packages for QLoRA.

### Guide: Persisting Data via Google Drive

By mounting Google Drive, we ensure that our training checkpoints and the final model weights are safely stored on the cloud. If the Colab runtime disconnects, we won't lose hours of training progress.

In [ ]:
# --- Google Drive Mounting ---
from google.colab import drive  # Colab utility to interact with Google Drive
drive.mount('/content/drive')    # Mount to /content/drive directory

### Guide: Installing the QLoRA Stack

We upgrade and install the Hugging Face AI ecosystem:
- `transformers`: For model architectures and the generation API.
- `trl`: To use `SFTTrainer` for easy Supervised Fine-Tuning.
- `peft`: For Low-Rank Adaptation (LoRA).
- `accelerate` & `bitsandbytes`: To handle GPU orchestration and 4-bit/8-bit quantization.

In [ ]:
# --- Install Required Libraries ---
# -q limits logs, -U upgrades to the newest valid versions
!pip install -q -U \
    transformers \
    trl \
    peft \
    accelerate \
    bitsandbytes

### Guide: Importing Modules and Secure Authentication

Google's Gemma models are **gated**, which means you must accept their terms of use on the Hugging Face website before downloading them. Once accepted, you authenticate here.

**Security Note:** Do not hardcode your token! We use Colab's built-in **Secrets Manager** (`userdata`). Navigate to the "🔑" icon on the left to add your `HF_TOKEN`.

In [ ]:
# --- Library Imports and Authentication ---
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig
from trl import SFTTrainer
from google.colab import userdata
from huggingface_hub import login

# Securely fetch the HF token and login
hf_token = userdata.get('HF_TOKEN')   # Best practice: Do not hardcode strings here
login(token=hf_token)                 # Authenticate to access gated Gemma models

---
## Phase 2: Model & Tokenizer Initialization
In this phase, we apply 4-bit precision to fit the model in the GPU and set up LoRA adapters so only a small fraction of the model's weights actually "train."

### Guide: 4-Bit Quantization

Using `BitsAndBytesConfig`, we load the base weights of `google/gemma-2b` in NormalFloat4 (`nf4`) precision while dynamically running calculations in `bfloat16`. This massively drops the memory requirement while retaining the model's raw reasoning ability.

In [ ]:
# --- Quantization Setup ---
model_id = "google/gemma-2b"                     # Target the authentic Gemma 2B base model

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                           # Only load the weights iteratively in 4-bit
    bnb_4bit_quant_type="nf4",                   # Optimal data type for weight quantization
    bnb_4bit_compute_dtype=torch.bfloat16,       # Computation happens in bfloat16 for stability
)

### Guide: Loading the Base Model and Tokenizer

We inject the config and map the model directly to the available GPU (`device_map="auto"`). 
We also explicitly set the `pad_token` because causal language models often don't have one configured by default out-of-the-box, which is needed when batch processing inputs of different lengths.

In [ ]:
# --- Load Tokenizer and Base Model ---
tokenizer = AutoTokenizer.from_pretrained(model_id)   # Download the target tokenizer
tokenizer.pad_token = tokenizer.eos_token             # Map padding to the EOS token for batch sequences

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,                   # Attach 4-bit config
    device_map="auto"                                 # Automatically distribute across the T4 GPU
)

---
## Phase 3: Dataset Preparation
We need a supervised dataset containing both questions and high-quality answers. We'll load the Alpaca dataset and structure it so the model can learn the standard instruction-response format.

### Guide: Loading & Formatting `alpaca`

Language models don't just magically figure out where an "answer" begins. We must string the instruction and response together using clear delimiters (like `### Instruction:` and `### Response:`).
We use `dataset.map()` to loop through the raw dataset and bind these elements into a single `text` column, which `SFTTrainer` will ingest.

In [ ]:
# --- Load and Format Dataset ---
# Fetch the standard Alpaca instruction-response dataset from Hugging Face
dataset = load_dataset("tatsu-lab/alpaca", split="train")

# Define formatting logic to merge 'instruction' and 'output' into one continuous string
def format_prompt(example):
    # A clean prompt block telling the base model exactly what context it is looking at
    return {
        "text": f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"
    }

# Apply the formatting function efficiently across the entire dataset
dataset = dataset.map(format_prompt)

---
## Phase 4: Training Configuration
Here we define our training metrics, learning rates, epochs, and finally wrap the model with LoRA using `SFTTrainer`.

### Guide: PEFT / LoRA Variables

LoRA freezes the 2.5 billion weights of Gemma and only injects highly efficient tiny matrices to train on. 
- `r`: The "rank" of the matrix. 16 is a balanced value.
- `target_modules`: Specific internal components (like `q_proj` and `v_proj` inside Attention layers) where we want to insert these trainable adapters.

In [ ]:
# --- Configure LoRA (Low-Rank Adaptation) ---
peft_config = LoraConfig(
    r=16,                                        # Rank matrix size (lower is faster, holds less capacity)
    lora_alpha=32,                               # Scaling constant (how strongly LoRA influences weights)
    lora_dropout=0.05,                           # Regularization: randomly shut down 5% of neural links to prevent overfitting
    bias="none",                                 # Don't bias the baseline operations (most stable)
    task_type="CAUSAL_LM",                       # Explicitly say we are training text generation
    target_modules=["q_proj", "v_proj"]          # Target the specific Query and Value attention heads of Gemma
)

### Guide: `TrainingArguments` setup

We set `bf16=True` since Gemma models inherently train very well in **bfloat16 precision** and it's natively supported on T4 GPUs. 
Notice the `optim="paged_adamw_8bit"`. This "paged" optimizer pushes optimizer memory back over to standard RAM if the GPU starts running out, which is a life-saver for Colab.

In [ ]:
# --- Define Training Hyperparameters ---
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/gemma-results", # Save securely to the mounted Drive
    per_device_train_batch_size=2,               # Number of simultaneous samples per pass
    gradient_accumulation_steps=4,               # Simulate an effective batch size of 8 (2*4) to save VRAM
    learning_rate=2e-4,                          # The standard, safe LR for QLoRA networks
    num_train_epochs=1,                          # One full pass through the Alpaca dataset
    logging_steps=10,                            # Print console logs to watch progress every 10 steps
    save_strategy="epoch",                       # Save our progress automatically after an epoch
    bf16=True,                                   # Employs bfloat16: ideal stable precision for T4/Gemma
    optim="paged_adamw_8bit"                     # Uses specialized optimizer to prevent GPU Memory errors
)

### Guide: Execute `SFTTrainer`

Everything clicks together here. The `SFTTrainer` takes the base model, applies the `peft_config` (LoRA), loads the formatted `alpaca` dataset natively mapped to the `"text"` variable via `dataset_text_field`, and starts training.

In [ ]:
# --- Initialization and Execution of Trainer ---
trainer = SFTTrainer(
    model=model,                                 # Base model
    train_dataset=dataset,                       # The parsed dataset with the 'text' key
    peft_config=peft_config,                     # Binding the LoRA configs
    dataset_text_field="text",                   # Explicitly specifying which column the trainer should digest
    args=training_args,                          # Attach hyperparams from earlier
    max_seq_length=512                           # Prevent excessively long prompts from crashing memory
)

# Start the actual training! (This will take a little while)
trainer.train()

---
## Phase 5: Inference & Testing
Once training has finished, we need to manually invoke the model to make sure it successfully learned to follow our "Instruction/Response" framework.

### Guide: Test Generation using the Base Model + LoRA weights
We structure our prompt string mathematically identical to how it trained (`### Instruction: ... \n\n ### Response:\n`).
The model is then forced into generating text to complete the pattern naturally, effectively "responding" to us!

In [ ]:
# --- Testing Fine-Tuned Output ---
prompt_question = "Explain machine learning in simple terms."

# Encase it in the structural prompt format
formatted_prompt = f"### Instruction:\n{prompt_question}\n\n### Response:\n"

# Tokenize text into tensor math array and shift it aggressively to the GPU
inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

# Generate predictive next tokens up to a strict limit
outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

# Decode raw numerical outputs back into human text, scrubbing the special tags like <s>
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

---
# Appendix: Adapting This Pipeline for Other Models

This pipeline works for **any causal LLM**. Three things typically change:

---

### A — LoRA Target Modules
| Model Family | `target_modules` |
|-------------|------------------|
| **Qwen 2.5** | `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj` |
| **LLaMA / TinyLlama** | `q_proj, v_proj` |
| **Mistral** | `q_proj, k_proj, v_proj, o_proj` |
| **Falcon** | `query_key_value` |

---

### B — Prompt Template
| Model | Template |
|-------|----------|
| **Qwen (ChatML)** | `<\|im_start\|>user\n{prompt}<\|im_end\|>` |
| **LLaMA 2 Chat** | `<s>[INST] {prompt} [/INST]` |
| **Alpaca** | `### Instruction:\n{prompt}\n### Response:` |

---

### C — Model Loader Class
| Model Type | Class |
|-----------|-------|
| **Causal LM** (GPT, LLaMA, Qwen, Mistral) | `AutoModelForCausalLM` |
| **Seq2Seq** (T5, FLAN-T5) | `AutoModelForSeq2SeqLM` |
| **Encoder** (BERT) | `AutoModelForSequenceClassification` |
